# Face Detector Comparison — YOLOv8n vs YOLOv11n vs RT-DETR

Trains three detectors on the **same** combined WIDER+FDDB dataset (`face_dataset.yaml`) so the comparison is apples-to-apples, then collects mAP / Precision / Recall / F1 **plus inference speed and model size** into one results table.

The face detector feeds the *face-presence* component of the Exam Behavior Index (EBI), and gates gaze/identity, so both **accuracy** and **real-time speed** matter.

**Before running:** same dataset zip in Drive as the fine-tune notebook (`MyDrive/knowing-eye-datasets.zip`).

**Runtime:** Runtime -> Change runtime type -> **T4 GPU**.

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Unzip dataset to local Colab disk (fast I/O) and point the yaml at it
import yaml

DATASET_ZIP = '/content/drive/MyDrive/knowing-eye-datasets.zip'
DATA_ROOT = '/content/datasets'

!mkdir -p {DATA_ROOT}
!unzip -q -o "{DATASET_ZIP}" -d {DATA_ROOT}

yaml_path = f'{DATA_ROOT}/face_dataset.yaml'
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)
cfg['path'] = DATA_ROOT
with open(yaml_path, 'w') as f:
    yaml.safe_dump(cfg, f)
print(cfg)

In [ ]:
# ---- Comparison config ----
# Keep EPOCHS identical across all models for a fair comparison.
# 100 matches the YOLOv8n baseline but is long (~1-2 hrs total on a T4).
# Drop to e.g. 50 if you want a faster first pass — just keep it the SAME for every model.
EPOCHS = 100
IMGSZ = 640

# (display_name, pretrained_weights, batch_size)
# RT-DETR is heavier, so it gets a smaller batch to fit the free-tier T4 (16 GB).
MODELS = [
    ('yolov8n', 'yolov8n.pt', 16),
    ('yolo11n', 'yolo11n.pt', 16),
    ('rtdetr-l', 'rtdetr-l.pt', 8),
]

import os
WEIGHTS_DIR = '/content/drive/MyDrive/knowing-eye-weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

In [ ]:
import shutil
import pandas as pd
from ultralytics import YOLO, RTDETR


def model_class(name):
    return RTDETR if name.startswith('rtdetr') else YOLO


rows = []
for name, weights, batch in MODELS:
    print(f'\n{"="*60}\nTRAINING {name}  (epochs={EPOCHS}, batch={batch})\n{"="*60}')
    Cls = model_class(name)
    try:
        model = Cls(weights)
        model.train(
            data=yaml_path, epochs=EPOCHS, imgsz=IMGSZ, batch=batch,
            patience=20, project='/content/runs', name=name,
            single_cls=True, exist_ok=True,
        )

        # Reload BEST weights and validate (so metrics reflect the best epoch)
        best = f'/content/runs/{name}/weights/best.pt'
        model = Cls(best)
        m = model.val(data=yaml_path, imgsz=IMGSZ)

        p = float(m.box.mp)   # mean precision
        r = float(m.box.mr)   # mean recall
        f1 = (2 * p * r / (p + r)) if (p + r) else 0.0
        inf_ms = float(m.speed.get('inference', float('nan')))  # ms / image
        fps = (1000.0 / inf_ms) if inf_ms else float('nan')
        n_params = sum(x.numel() for x in model.model.parameters())

        rows.append({
            'model': name,
            'mAP50 (%)': round(m.box.map50 * 100, 2),
            'mAP50-95 (%)': round(m.box.map * 100, 2),
            'Precision (%)': round(p * 100, 2),
            'Recall (%)': round(r * 100, 2),
            'F1 (%)': round(f1 * 100, 2),
            'Inference (ms)': round(inf_ms, 2),
            'FPS': round(fps, 1),
            'Params (M)': round(n_params / 1e6, 2),
        })

        # Persist weights so they survive the session
        shutil.copy2(best, f'{WEIGHTS_DIR}/{name}_best.pt')
        print(f'[{name}] done -> {rows[-1]}')
    except Exception as e:
        print(f'[{name}] FAILED: {e}')
        rows.append({'model': name, 'error': str(e)})

    # Save partial results after each model in case a later one crashes
    pd.DataFrame(rows).to_csv(f'{WEIGHTS_DIR}/comparison_results.csv', index=False)

In [ ]:
# Final comparison table
df = pd.DataFrame(rows)
df.to_csv(f'{WEIGHTS_DIR}/comparison_results.csv', index=False)
print('Saved to', f'{WEIGHTS_DIR}/comparison_results.csv')
df

## After it finishes

Download `comparison_results.csv` (and the `*_best.pt` weights) from `MyDrive/knowing-eye-weights/`.

Send me `comparison_results.csv` and I'll build the paper-style comparison figure + discussion HTML (matching `yolo-model-testing.html`), including the accuracy-vs-speed trade-off chart that justifies which detector to wire into the EBI face-presence stage.